# Random Forest Classifier
## Real-world scenario: Detecting fraudulent transactions

A payment company wants to flag whether a transaction is **fraudulent (1)** or **genuine (0)**. A Random Forest combines many decision trees and **votes**, which usually beats a single tree and is more robust - ideal for messy real-world fraud data.

### Step 1 - Import the libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

np.random.seed(42)

### Step 2 - Create a small, realistic dataset
80 transactions with amount, time of day, distance from home and number of recent transactions.

In [ ]:
n = 80
amount        = np.random.randint(5, 5000, n)
hour          = np.random.randint(0, 24, n)           # hour of the day
distance_km   = np.random.randint(0, 500, n)          # from usual location
recent_txns   = np.random.randint(1, 20, n)           # txns in last hour

# Large amount + far away + odd hour + many recent txns -> more suspicious
fraud_score = (amount / 5000) + (distance_km / 500) \
              + ((hour < 6).astype(int) * 0.4) + (recent_txns / 20) \
              + np.random.normal(0, 0.3, n)
is_fraud = (fraud_score > fraud_score.mean() + 0.3).astype(int)

df = pd.DataFrame({
    'amount': amount, 'hour': hour, 'distance_km': distance_km,
    'recent_txns': recent_txns, 'is_fraud': is_fraud
})

# Messy data on purpose
df.loc[6, 'amount'] = np.nan
df.loc[11, 'distance_km'] = np.nan
df = pd.concat([df, df.iloc[[3]]], ignore_index=True)
df.head()

### Step 3 - Explore the data

In [ ]:
print('Shape:', df.shape)
print('\nMissing:\n', df.isnull().sum())
print('\nDuplicates:', df.duplicated().sum())
print('\nFraud vs genuine:\n', df['is_fraud'].value_counts())

### Step 4 - Clean the data

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
df['amount'] = df['amount'].fillna(df['amount'].median())
df['distance_km'] = df['distance_km'].fillna(df['distance_km'].median())
print('Missing after cleaning:', df.isnull().sum().sum())

### Step 5 - Features (X) and target (y)

In [ ]:
X = df[['amount', 'hour', 'distance_km', 'recent_txns']]
y = df['is_fraud']

### Step 6 - Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

### Step 7 - Train the Random Forest
`n_estimators=100` means 100 trees vote on each prediction.

In [ ]:
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

### Step 8 - Evaluate

In [ ]:
y_pred = model.predict(X_test)
print('Accuracy:', round(accuracy_score(y_test, y_pred), 3))
print('\nConfusion matrix:\n', confusion_matrix(y_test, y_pred))
print('\nReport:\n', classification_report(y_test, y_pred, zero_division=0))

### Step 9 - Which features matter most?
Random Forests can tell us how important each feature was to the decision.

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns)
importances = importances.sort_values()
importances.plot(kind='barh')
plt.title('Feature importance for fraud detection')
plt.xlabel('Importance'); plt.show()